# Domain Wall Coarsening — 3D Ising Model

Analyses the decay of domain wall density after an instantaneous quench
from high T (disordered) to low T (deep ordered phase).

Theory predicts Allen-Cahn coarsening: rho(t) ~ t^(-z)
- 2D: z = 1/2
- 3D: z = 1/3

**Generate data first:**
```bash
cargo run --release --bin coarsening -- --n 30 --steps 50000 --sample-every 100 --outdir analysis/data
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DATA_DIR = Path('data')

files = sorted(DATA_DIR.glob('coarsening_*.csv'))
print('Found:', [f.name for f in files])

if not files:
    raise SystemExit('No coarsening data found. Generate data first.')

dfs = {f.stem: pd.read_csv(f) for f in files}
for name, df in dfs.items():
    print(f'{name}: {len(df)} points, t=[{df.t.min()}, {df.t.max()}], rho=[{df.rho.min():.4f}, {df.rho.max():.4f}]')

## 1. Domain wall density vs time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, df in dfs.items():
    axes[0].plot(df['t'], df['rho'], label=name)
    mask = df['t'] > 0
    axes[1].loglog(df.loc[mask, 't'], df.loc[mask, 'rho'], label=name)

axes[0].set_xlabel('t (sweeps)')
axes[0].set_ylabel('Domain wall density rho')
axes[0].set_title('Linear scale')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('t (sweeps)')
axes[1].set_ylabel('rho')
axes[1].set_title('Log-log scale')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('coarsening_raw.png', bbox_inches='tight')
plt.show()

## 2. Fit coarsening exponent z

Fit log(rho) = -z * log(t) + const in the power-law regime.
Skip first 10% of steps (transient).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
z_values = {}

for name, df in dfs.items():
    sub = df[(df['t'] > df['t'].max() * 0.1) & (df['t'] > 0)].copy()
    slope, intercept, r, p, se = stats.linregress(np.log(sub['t']), np.log(sub['rho']))
    z = -slope
    z_values[name] = z
    print(f'{name}: z = {z:.4f}  (R2={r**2:.4f})')
    print(f'  3D Allen-Cahn theory: z = {1/3:.4f}')

    ax.loglog(sub['t'], sub['rho'], label=name)
    t_fit = np.array([sub['t'].min(), sub['t'].max()])
    ax.loglog(t_fit, np.exp(intercept + slope * np.log(t_fit)), 'k--',
              label=f'fit z={z:.3f}')

# Theory reference lines
sub_last = list(dfs.values())[-1]
sub_last = sub_last[sub_last['t'] > 0]
t_ref = np.logspace(np.log10(sub_last['t'].quantile(0.1)), np.log10(sub_last['t'].max()), 100)
mid = len(t_ref) // 2
ax.loglog(t_ref, sub_last['rho'].median() * (t_ref / t_ref[mid])**(-1/3), 'r:', label='3D theory z=1/3')
ax.loglog(t_ref, sub_last['rho'].median() * (t_ref / t_ref[mid])**(-1/2), 'b:', label='2D theory z=1/2')

ax.set_xlabel('t (sweeps)')
ax.set_ylabel('Domain wall density rho')
ax.set_title('Coarsening exponent fit')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('coarsening_fit.png', bbox_inches='tight')
plt.show()

## 3. Summary table

In [ ]:
Z_3D = 1/3
print(f'{"dataset":<35} {"z_measured":>12} {"3D theory":>12} {"error %":>10}')
print('-' * 72)
for name, z in z_values.items():
    err = abs(z - Z_3D) / Z_3D * 100
    print(f'{name:<35} {z:>12.4f} {Z_3D:>12.4f} {err:>9.1f}%')